In [ ]:
!pip install yfinance pandas tabulate tqdm -q

In [ ]:
import yfinance as yf
import pandas as pd
import time
import warnings
from tqdm.notebook import tqdm
warnings.filterwarnings('ignore')

def get_metrics(ticker):
    try:
        info = yf.Ticker(ticker.strip().upper()).info
        if not info.get('regularMarketPrice'):
            return None
        def safe(v, div=1):
            try: return round(v / div, 2) if v else None
            except: return None
        return {
            'Ticker': ticker.upper(),
            'PE_trailing': info.get('trailingPE'),
            'PE_forward': info.get('forwardPE'),
            'PEG': info.get('pegRatio'),
            'PB': info.get('priceToBook'),
            'PS': info.get('priceToSalesTrailing12Months'),
            'EV_EBITDA': info.get('enterpriseToEbitda'),
            'ROE': safe(info.get('returnOnEquity'), 0.01),
            'RevGrowth': safe(info.get('revenueGrowth'), 0.01),
        }
    except:
        return None

def analyze_in_batches(tickers, batch_size=25):
    print(f"Processing {len(tickers)} tickers in batches of {batch_size}...")
    results = []
    num_batches = (len(tickers) + batch_size - 1) // batch_size
    for i in tqdm(range(0, len(tickers), batch_size), desc="Batches"):
        batch = tickers[i:i+batch_size]
        for t in batch:
            res = get_metrics(t)
            if res: results.append(res)
        time.sleep(1.0)
        current_batch = (i // batch_size) + 1
        print(f"Done batch {current_batch}/{num_batches}")
    df = pd.DataFrame(results)
    if df.empty: return df
    for col in ['PE_trailing','PE_forward','PEG','PB','PS','EV_EBITDA']:
        if col in df.columns:
            med = df[col].median()
            if med and med > 0:
                df[col + '_vsMed'] = (df[col] / med).round(2)
    return df

# ==================== CHANGE THIS ====================
tickers = ['MSFT','GOOGL','AMZN','META','AAPL','NVDA','TSLA','AMD']  # paste your list here
# ===================================================

df = analyze_in_batches(tickers)

if not df.empty:
    print("\n" + "="*90)
    print("VALUATION TABLE")
    print("="*90)
    print(df.to_markdown(index=False))
    
    prompt = f"""You are a strict value investor.\nAnalyze which stocks are cheap vs the group (vsMed < 1 = cheaper).\n\nData:\n{df.to_markdown(index=False)}"""
    print("\n" + "="*90)
    print("COPY THIS PROMPT:")
    print("="*90)
    print(prompt)
